In [1]:
import os
from pathlib import Path
# Change cwd to the project root (parent of 'notebooks/')
os.chdir(Path.cwd().parent)
Path.cwd()

PosixPath('/Users/jbrandt/code/birddog')

In [2]:
# uncomment to use hosted db
del os.environ["BIRDDOG_USE_LOCAL_NOCODB"]
if os.environ.get("BIRDDOG_USE_LOCAL_NOCODB"):
    print("using local nocodb")
else:
    print("using aws nocodb")

using aws nocodb


In [3]:
from birddog.database import Database

2026-08-04 18:45:07,704 [INFO] Using aws nocodb api: http://nocodb-env.eba-xhmfyydr.us-east-2.elasticbeanstalk.com


In [4]:
db = Database()

2026-08-04 18:45:08,834 [INFO] creating NocoDBDatabase(host=http://nocodb-env.eba-xhmfyydr.us-east-2.elasticbeanstalk.com, base_id=prsjtz30iuhk88f) instance
2026-08-04 18:45:09,045 [INFO] 
service throttle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight
  ----------------------------------------------------------------------------------------
  nocodb.internal:api                   20.00     4.78    39.00       0.00           24


In [18]:
def orphan_scan(db, limit=100):
    orphans, _ = db.scan(
        "Documents", 
        view_name="BD:Orphan Documents", 
        fields=["sha1_hash","url", "title", "num_owners", "owning_pages"],
        limit=limit)
    return orphans

In [19]:
def sha1_lookup(db, sha1_hash):
    recs, _ = db.scan(
        "Documents", 
        where=("sha1_hash", "eq", sha1_hash), 
        fields=[
            "url", 
            "title",
            "num_owners", 
            "owning_pages"])
    return recs

In [7]:
def find_dupes(db, records):
    for rec in records:
        result = {}
        sha1_hash = rec.get("sha1_hash")
        if sha1_hash:
            recs = sha1_lookup(db, sha1_hash)
            if len(recs) > 1:
                result[sha1_hash] = recs
        return result

In [8]:
def do_orphan_batch(db):
    dupes = find_dupes(db, orphan_scan(db))
    return dupes

In [9]:
do_orphan_batch(db)

{}

In [20]:
sha1_lookup(db, "b58a3295c3479991ae0aeb66a38473f95fdd20d9")

2026-08-04 18:50:46,217 [INFO] 
service throttle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight
  ----------------------------------------------------------------------------------------
  nocodb.internal:api                   25.00     0.02    39.00       0.00           24


[{'Id': 457905,
  'title': 'File:ДАХмО_Р-582-2-272_Особова_справа_Блатман_С._Й._(1920-1920).pdf',
  'url': 'https://commons.wikimedia.org/wiki/File:ДАХмО_Р-582-2-272_Особова_справа_Блатман_С._Й._(1920-1920).pdf',
  'num_owners': 1,
  'owning_pages': [{'Id': 796709, 'title': 'ДАХмО/Р-582/2/272'}]},
 {'Id': 457907,
  'title': 'Файл:ДАХмО_Р-582-2-272_Особова_справа_Блатман_С._Й._(1920-1920).pdf',
  'url': 'https://uk.wikisource.org/wiki/File:ДАХмО_Р-582-2-272_Особова_справа_Блатман_С._Й._(1920-1920).pdf',
  'num_owners': 1,
  'owning_pages': [{'Id': 796709, 'title': 'ДАХмО/Р-582/2/272'}]}]

In [21]:
sha1_lookup(db, "5f0fb0c032d9697c67d844d3bb46c3c952654253")

[{'Id': 328490,
  'title': 'File:ДАХмО_Р-582-2-272_Особова_справа_Блатман_С._Й._(1920).pdf',
  'url': 'https://uk.wikisource.org/wiki/File:ДАХмО_Р-582-2-272_Особова_справа_Блатман_С._Й._(1920).pdf',
  'num_owners': 0,
  'owning_pages': []}]